# 04_03_Loading Benchmark to SQL Server

In [1]:
import os
from pathlib import Path

import pandas as pd

import infrabel_punctuality as ip

In [2]:
PATH_PROCESSED = Path("../data/processed/")

# Table of Contents  

- [3. LOADING BENCHMARK](#3-loading-benchmark)

- [3.1. Preparing SQL Server Connection](#31-preparing-sql-server-connection)  

- [3.2. Loading DataFrame from `processed` Directory](#32-loading-dataframe-from-processed-directory)  

- [3.3. Loading `Infrabel_Benchmark` to SQL Server](#33-loading-infrabel_benchmark-to-sql-server)  

## 3. LOADING BENCHMARK

**LOADING STRATEGY**  

**This notebook loads the prepared benchmark DataFrame into the SQL Server table: `ref.Infrabel_Benchmark`.**  

1) To avoid implicit type conversions between the DataFrame dtypes and the SQL column types, and to ensure better control over column typing, **the benchmark table has already been created** using an SQL DDL script: `03_05_create_ref_infrabel_benchmark` in `/sql/ddl/`.  

2) The loading strategy is a **full load**. Before each load run, a **TRUNCATE TABLE** statement is executed on the target tables to prevent duplicates while preserving the DDL-defined table structure.  

3) The loading process is performed through **SQLAlchemy** using the functions provided by the `sql_server_connection` module of the project's `infrabel_punctuality` package.  

4) The SQLAlchemy engine is configured to connect to the project's SQL Server database, `infrabel_punctuality_dwh`, created in *01_create_database* located in `/sql/ddl/`. The **database name** is therefore hardcoded in this notebook.  

5) The engine is also configured to connect to a local SQL Server instance using `localhost` as the default **server name**. If the SQL Server instance on your machine is not accessible through `localhost`, set the `SQL_SERVER` environment variable to the correct server or instance name.


## 3.1. Preparing SQL Server Connection

In this section:  

* The SQLAlchemy engine is created with the required parameters - driver, server and database names - and its connection to the database is tested.  

* The `select_sql_driver()` and `get_engine()` functions support both *ODBC Driver 17 for SQL Server* and *ODBC Driver 18 for SQL Server*.  

In [3]:
driver = ip.select_sql_driver()
server = os.getenv("SQL SERVER", "localhost")
database = "infrabel_punctuality_dwh"

Driver: ODBC Driver 18 for SQL Server


In [4]:
engine = ip.get_engine(driver, server, database)

In [5]:
ip.test_connection(engine)

Connection successful to database: infrabel_punctuality_dwh


## 3.2. Loading DataFrame from `processed` Directory

In [6]:
Infrabel_Benchmark = pd.read_parquet(PATH_PROCESSED / "infrabel_benchmark_prepared.parquet")
Infrabel_Benchmark.head()

,Year,Month,Punctuality_Pct,Train_Count,Train_6Min_Late_Count,Delay_Minutes
0,2026,2026-05-01,92.39,109777,101421,184749
1,2026,2026-04-01,93.40,113704,106197,165290
2,2026,2026-03-01,93.05,112550,104728,181743
3,2026,2026-02-01,93.25,108663,101330,162212
4,2026,2026-01-01,92.39,107866,99662,178328


## 3.3. Loading `Infrabel_Benchmark` to SQL Server

In [7]:
ip.full_load_to_sql_server(
    engine,
    dataframe=Infrabel_Benchmark,
    table_name="Infrabel_Benchmark",
    schema="ref"
)